# State Migration - 02: Additive vs breaking changes

> **MLCourse - Agentic AI - LangGraph - Module 10**

Notebook 01 showed that a schema change can break live threads. It did not say
*which* changes do. Not every edit to a `TypedDict` is dangerous, and knowing
the difference is what lets you ship most changes without ceremony and treat
the rest with respect.

This notebook builds the taxonomy by **running each kind of change** against a
checkpoint written by an older schema, and recording what actually happens.

### What you will learn

1. Which schema changes are **additive** (old checkpoints keep working).
2. Which are **breaking** (old checkpoints crash).
3. Which are **silently breaking** - no error, wrong answer. These are the
   dangerous ones.
4. A checklist you can apply to a pull request before it ships.

Still no LLM calls and no API key.

### Setup


In [ ]:
import operator
import os
import sqlite3
from typing import Annotated, List, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

DB_PATH = "taxonomy_demo.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))

# A tiny helper so each experiment below reads the same way: build a one-node
# graph over a state class, run it on a thread, report what happened.
def run(state_cls, node_fn, thread_id, payload, label):
    """Compile a one-node graph over `state_cls` and invoke it on `thread_id`."""
    b = StateGraph(state_cls)
    b.add_node("work", node_fn)
    b.add_edge(START, "work")
    b.add_edge("work", END)
    graph = b.compile(checkpointer=checkpointer)
    cfg = {"configurable": {"thread_id": thread_id}}
    try:
        result = graph.invoke(payload, cfg)
        print(f"  {label}: OK -> {result}")
        return graph, cfg, result
    except Exception as e:
        print(f"  {label}: {type(e).__name__}: {e}")
        return graph, cfg, None


print("taxonomy experiments -- each writes a V1 checkpoint, then runs V2 on it")


### Case A - Adding a field, read defensively → **ADDITIVE (safe)**

The change: a new field appears in the state class, and the code that reads it
uses `.get()` with a sensible default.

This is the change you want to be making. It works on old checkpoints because
the code never assumes the field is present.

### Case A: additive field, read with .get()


In [ ]:
class A_V1(TypedDict):
    order_id: str
    status: str


class A_V2(TypedDict):
    order_id: str
    status: str
    priority: str          # NEW


def a_node_v1(state) -> dict:
    return {"status": "v1-done"}


def a_node_v2(state) -> dict:
    # THE SAFE PATTERN: .get() with a default that means the same thing the
    # system meant before the field existed. Old orders had no priority
    # concept at all, so "normal" is the honest translation.
    priority = state.get("priority", "normal")
    return {"status": f"v2-done:{priority}"}


print("Case A -- additive field, defensive read")
run(A_V1, a_node_v1, "case-a", {"order_id": "A-1", "status": "new"}, "V1 write")
run(A_V2, a_node_v2, "case-a", {"order_id": "A-1"}, "V2 on old checkpoint")


**Result: safe.** The old thread ran to completion and got `priority=normal`.

Two things made this work, and you need both:

1. The field was **added**, not renamed or retyped. Nothing that already
   existed changed meaning.
2. The reader used `.get()` with a default. `state["priority"]` would have
   raised `KeyError` on exactly this thread - which is Case B.

> **`.get()` is not a magic fix.** It is a decision that a missing field has a
> defined meaning. If no default is *correct* - say the new field is
> `customer_consent` and you cannot invent one - then `.get()` is worse than a
> crash, and you need a real migration instead.

### Case B - Adding a field, read directly → **BREAKING (loud)**

The same schema change, one character different in the node. This is the
failure from notebook 01, included here so the taxonomy is complete.

### Case B: additive field, direct read


In [ ]:
def b_node_v2(state) -> dict:
    return {"status": f"v2-done:{state['priority']}"}     # direct [] access


print("Case B -- additive field, direct read")
run(A_V1, a_node_v1, "case-b", {"order_id": "B-1", "status": "new"}, "V1 write")
run(A_V2, b_node_v2, "case-b", {"order_id": "B-1"}, "V2 on old checkpoint")


**Result: breaking, but loudly.** A `KeyError` names the exact field.

Loud failures are the *good* kind. You get a stack trace pointing at the
problem, and the thread's stored data is untouched - fix the code, redeploy,
and the thread continues. Compare with Case C.

### Case C - Renaming a field → **SILENTLY BREAKING**

The change: `user_name` becomes `username`. Purely cosmetic, obviously
harmless, the kind of tidy-up that sails through code review.

Watch what happens to the data.

### Case C: renamed field


In [ ]:
class C_V1(TypedDict):
    user_name: str          # old name
    visits: int


class C_V2(TypedDict):
    username: str           # renamed!
    visits: int


def c_node_v1(state) -> dict:
    return {"visits": state["visits"] + 1}


def c_node_v2(state) -> dict:
    # Defensive read, exactly as Case A recommended.
    return {"visits": state["visits"] + 1,
            "username": state.get("username", "<MISSING>")}


print("Case C -- renamed field")
g1, cfg_c, _ = run(C_V1, c_node_v1, "case-c",
                   {"user_name": "ada", "visits": 0}, "V1 write")
print(f"    state after V1: {g1.get_state(cfg_c).values}")

g2, _, _ = run(C_V2, c_node_v2, "case-c", {"visits": 5}, "V2 on old checkpoint")
print(f"    state after V2: {g2.get_state(cfg_c).values}")


### Result: no exception, and the data is gone

Read those two state lines again.

```
after V1:  {'user_name': 'ada', 'visits': 1}
after V2:  {'username': '<MISSING>', 'visits': 6}
```

`ada` is **gone**. Not moved, not defaulted - absent from the state the new
graph can see. And nothing raised. The graph ran, the counter incremented
correctly, and a downstream node would now cheerfully greet this returning
customer as `<MISSING>`.

Two mechanisms combined to hide this:

1. **Channels are addressed by name.** `user_name` and `username` are simply
   different channels. The new graph does not declare `user_name`, so it never
   loads it. The old value is still in the database, orphaned.
2. **The defensive `.get()` from Case A masked the symptom.** It converted a
   loud `KeyError` into a plausible-looking default. The advice from Case A is
   still correct; it just does not protect against this class of change.

This is why renames deserve more caution than additions, despite looking
smaller. **A rename is a delete plus an add**, and the delete half is
invisible.

> The same logic applies to **removing** a field: the data stays in the
> database as an orphaned channel, consuming space and quietly reappearing if
> anyone ever re-adds a field with that name.

### Case D - Changing a field's type or reducer → **BREAKING, and worse than it looks**

The change: `log` was a string; now it is a list with an `operator.add`
reducer so entries accumulate. A very common evolution - a field that held one
value now needs to hold several.

### Case D: type + reducer change


In [ ]:
class D_V1(TypedDict):
    log: str                                        # a single string


class D_V2(TypedDict):
    log: Annotated[List[str], operator.add]         # now an accumulating list


def d_node_v1(state) -> dict:
    return {"log": "first entry"}


def d_node_v2(state) -> dict:
    return {"log": ["second entry"]}


print("Case D -- type and reducer change")
g1, cfg_d, _ = run(D_V1, d_node_v1, "case-d", {"log": ""}, "V1 write")
print(f"    state after V1: {g1.get_state(cfg_d).values}")

g2, _, _ = run(D_V2, d_node_v2, "case-d", {"log": ["x"]}, "V2 on old checkpoint")


In [6]:
# ---- Case D, continued: the thread is now unreadable, not just unrunnable --
# In the other cases we could still inspect the thread's state to decide what
# to do about it. Here, even reading it fails.

print("Can we at least LOOK at the broken thread's state?")
try:
    print("  ", g2.get_state(cfg_d).values)
except Exception as e:
    print(f"  {type(e).__name__}: {e}")
    print("\n  get_state() applies the reducer to replay stored writes, and")
    print("  the reducer is the thing that is broken. So inspection fails too.")

print("\nBut the raw checkpoint is still readable underneath:")
raw = next(checkpointer.list(cfg_d))
print(f"  {raw.checkpoint['channel_values']}")
print("\n  The data was never lost -- only the graph's interpretation of it")
print("  is broken. That is what makes recovery possible (notebook 04).")

Can we at least LOOK at the broken thread's state?
  TypeError: can only concatenate str (not "list") to str

  get_state() applies the reducer to replay stored writes, and
  the reducer is the thing that is broken. So inspection fails too.

But the raw checkpoint is still readable underneath:
  {'log': 'first entry', '__start__': {'log': ['x']}}

  The data was never lost -- only the graph's interpretation of it
  is broken. That is what makes recovery possible (notebook 04).


### Result: breaking at a deeper level

`TypeError: can only concatenate str (not "list") to str` - thrown from inside
LangGraph's channel machinery, not from your node. The stored value is a
string, the reducer is `operator.add` expecting lists, and `"first entry" +
["second entry"]` is not a thing.

What makes this case distinct from Case B is the **blast radius**:

| | Case B (`KeyError`) | Case D (reducer mismatch) |
|---|---|---|
| Can the graph run? | no | no |
| Can you call `get_state()`? | **yes** | **no** |
| Can you inspect via the raw checkpointer? | yes | yes |
| Fix without touching data? | yes - fix the node | **no** - the data must change |

Case D corrupts your ability to *observe* the thread through the normal API,
because `get_state()` replays pending writes through the reducer to build the
snapshot. Your debugging tool is broken by the same bug you are debugging.

The good news is in that last cell: the raw checkpoint is intact. The bytes
were never lost - only the current schema's interpretation of them. Notebook
04 uses exactly this to repair threads.

> **Reducer changes are the most under-estimated schema change in LangGraph.**
> Adding `Annotated[..., operator.add]` to an existing field looks like a
> type-hint tweak. It changes how every future write merges with stored data.

### The taxonomy

Everything above, in the order you should worry about it:

| # | Change | Verdict | What happens to old threads |
|---|---|---|---|
| A | Add a field, read with `.get(default)` | ✅ **additive** | works; field takes the default |
| B | Add a field, read with `state[...]` | ❌ breaking, **loud** | `KeyError` naming the field |
| C | **Rename** a field | ⚠️ breaking, **silent** | old value orphaned; new field empty; **no error** |
| - | **Remove** a field | ⚠️ silent | value orphaned in the DB, invisible |
| D | Change type or **reducer** | ❌ breaking, **deep** | `TypeError` in the channel; `get_state()` also fails |
| - | Add/remove a **node** | ✅ usually additive | state is unaffected; but see the note below |
| - | Change **node names** | ⚠️ risky | a suspended thread's pending task points at a node name that no longer exists |

### The two rules that follow from this table

**Rule 1 - additive changes are cheap; everything else is a migration.**
Adding a field and reading it defensively needs no ceremony. A rename, a
removal, a retype, or a reducer change is a *data* change and needs a
migration function (notebook 03) plus a plan for existing checkpoints
(notebook 04).

**Rule 2 - prefer loud breakage to silent breakage.** Case B is a better
outcome than Case C. If you must make a risky change, make it fail fast: keep
the old field in the schema temporarily and assert on it, rather than
renaming and hoping.

### The rename you should do instead

Never rename in one step. Do it in three deploys:

1. **Add** `username` alongside `user_name`. Write both, read `username` with
   a fallback to `user_name`.
2. **Backfill** existing checkpoints so every thread has `username`
   (notebook 04).
3. **Remove** `user_name` once no checkpoint depends on it.

This is the expand-migrate-contract pattern from database schema management,
and it applies here for exactly the same reason: you cannot update the code
and the data at the same instant.

### A pull-request checklist

Before merging any change to a graph's state class, ask:

- [ ] Did I **add** fields only? → likely safe; confirm every read uses
      `.get()` with a default that is *correct*, not merely convenient.
- [ ] Did I **rename or remove** a field? → silent data loss. Use
      expand-migrate-contract.
- [ ] Did I change a field's **type** or its **reducer**? → old checkpoints
      become unreadable through `get_state()`. Migrate the data.
- [ ] Did I rename a **node**? → suspended threads hold pending tasks naming
      the old node. Drain them before deploying, or keep an alias.
- [ ] Are any threads currently **suspended at an `interrupt()`**? → those are
      the ones that break first and cannot be restarted. Check before you
      deploy, not after.
- [ ] Does my state carry a **version number**? → if not, start now. That is
      notebook 03.

### Key takeaways

- **Additive** changes (new field + defensive read) are safe on old
  checkpoints. Everything else is a data migration.
- **Loud** breakage (`KeyError`) is much better than **silent** breakage.
  Renames and removals produce no error and lose data.
- Channels are addressed **by name**, so a rename orphans the old value rather
  than moving it. Measured above: `user_name='ada'` vanished with no exception.
- **Reducer changes** break at the channel level, and they break `get_state()`
  too - your inspection tool fails alongside the graph. The raw checkpoint
  remains readable, which is what makes repair possible.
- Do renames as **expand → migrate → contract**, never in one deploy.

**Next:** `03_versioned_state_and_migrations.ipynb` - putting a version stamp
in state and writing a migration chain that upgrades old checkpoints on the
way in.